In [1]:
from __future__ import annotations

from pathlib import Path
import random
import re

import numpy as np
import pandas as pd
from scipy.stats import kendalltau
from sentence_transformers import CrossEncoder, InputExample
from torch.utils.data import DataLoader

# ------------------------------
# Config
# ------------------------------

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

BASE = Path(r"/teamspace/studios/this_studio/QA")
DATA_DIR = BASE / "data"
OUT_DIR = BASE / "submissions"
OUT_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_PATH = DATA_DIR / "Mysterious_Affair_at_Styles_Train_Data.csv"
BOOK_A_PATH = DATA_DIR / "BookA_test.csv"
BOOK_B_PATH = DATA_DIR / "BookB_test.csv"

BOOK_A_OUT = OUT_DIR / "BookA_rating.csv"
BOOK_B_OUT = OUT_DIR / "BookB_rating.csv"

CROSS_ENCODER_NAME = "cross-encoder/stsb-roberta-base"
DO_FINE_TUNE = True
EPOCHS = 1
BATCH_SIZE = 8
HARD_NEG_PER_POS = 2
NUM_MATCHES = 10000
K_FACTOR = 24.0

# ------------------------------
# Text helpers
# ------------------------------

def clean_text(text: str) -> str:
    text = "" if pd.isna(text) else str(text)
    text = text.replace("\u2019", "'").replace("\u201c", '"').replace("\u201d", '"')
    text = re.sub(r"\s+", " ", text).strip()
    return text

def split_sentences(text: str) -> list[str]:
    text = clean_text(text)
    return [part.strip() for part in re.split(r"(?<=[.!?])\s+", text) if part.strip()]

def head_snippet(text: str, n_sentences: int = 3, max_chars: int = 900) -> str:
    sentences = split_sentences(text)
    snippet = " ".join(sentences[:n_sentences]) if sentences else clean_text(text)
    return snippet[:max_chars]

def tail_snippet(text: str, n_sentences: int = 3, max_chars: int = 900) -> str:
    sentences = split_sentences(text)
    snippet = " ".join(sentences[-n_sentences:]) if sentences else clean_text(text)
    return snippet[-max_chars:]

def read_frame(path: Path) -> pd.DataFrame:
    frame = pd.read_csv(path)
    if "text" not in frame.columns:
        raise ValueError(f"Expected a text column in {path.name}")
    frame["text"] = frame["text"].map(clean_text)
    if "page" not in frame.columns:
        frame["page"] = np.arange(1, len(frame) + 1, dtype=int)
    return frame

def page_order(frame: pd.DataFrame, order_idx: list[int]) -> pd.DataFrame:
    return pd.DataFrame(
        {
            "original_page": np.arange(1, len(order_idx) + 1, dtype=int),
            "shuffled_page": frame.iloc[order_idx]["page"].to_numpy(dtype=int),
        }
    )

# ------------------------------
# Load data
# ------------------------------

print("Loading data...")
train_df = read_frame(TRAIN_PATH)
book_a_df = read_frame(BOOK_A_PATH)
book_b_df = read_frame(BOOK_B_PATH)

# ------------------------------
# Cross-encoder training data
# ------------------------------

def build_pairs(df: pd.DataFrame, hard_neg_per_pos: int = 2) -> list[InputExample]:
    texts = df["text"].tolist()
    n = len(texts)
    examples: list[InputExample] = []

    # Positive adjacent examples: A should come before B.
    for i in range(n - 1):
        examples.append(InputExample(texts=[tail_snippet(texts[i]), head_snippet(texts[i + 1])], label=1.0))

    # Hard negatives: same local context, but wrong continuation.
    for i in range(n - 1):
        candidates = [j for j in range(n) if j not in {i, i + 1} and abs(j - (i + 1)) <= 6]
        if not candidates:
            continue
        chosen = random.sample(candidates, k=min(hard_neg_per_pos, len(candidates)))
        for j in chosen:
            examples.append(InputExample(texts=[tail_snippet(texts[i]), head_snippet(texts[j])], label=0.0))

    random.shuffle(examples)
    return examples

print("Loading cross-encoder...")
judge_model = CrossEncoder(CROSS_ENCODER_NAME, num_labels=1)

if DO_FINE_TUNE:
    print("Fine-tuning cross-encoder on training pairs...")
    train_examples = build_pairs(train_df, hard_neg_per_pos=HARD_NEG_PER_POS)
    train_loader = DataLoader(train_examples, shuffle=True, batch_size=BATCH_SIZE)
    judge_model.fit(
        train_dataloader=train_loader,
        epochs=EPOCHS,
        warmup_steps=max(1, len(train_loader) // 10),
        show_progress_bar=True,
    )

# ------------------------------
# Judge and Elo tournament
# ------------------------------

def judge_probability(text_a: str, text_b: str) -> float:
    score = judge_model.predict([(tail_snippet(text_a), head_snippet(text_b))], show_progress_bar=False)[0]
    prob = float(1.0 / (1.0 + np.exp(-score)))
    return float(np.clip(prob, 1e-4, 1.0 - 1e-4))

def elo_expected(rating_a: float, rating_b: float) -> float:
    return 1.0 / (1.0 + 10.0 ** ((rating_b - rating_a) / 400.0))

def elo_update(rating_a: float, rating_b: float, score_a: float, k_factor: float = K_FACTOR) -> tuple[float, float]:
    expected_a = elo_expected(rating_a, rating_b)
    expected_b = 1.0 - expected_a
    rating_a = rating_a + k_factor * (score_a - expected_a)
    rating_b = rating_b + k_factor * ((1.0 - score_a) - expected_b)
    return rating_a, rating_b

def elo_tournament(texts: list[str], num_matches: int = NUM_MATCHES, initial_rating: float = 1200.0) -> np.ndarray:
    n = len(texts)
    ratings = np.full(n, initial_rating, dtype=np.float64)

    if n < 2:
        return ratings

    for match_idx in range(num_matches):
        i, j = random.sample(range(n), 2)

        # Ask the judge whether text i should come before text j.
        p_i_before_j = judge_probability(texts[i], texts[j])

        # Fractional Elo update uses the predicted probability directly.
        ratings[i], ratings[j] = elo_update(ratings[i], ratings[j], p_i_before_j)

        # Optional symmetry pass every few matches to stabilize ratings.
        if match_idx % 5 == 0:
            p_j_before_i = judge_probability(texts[j], texts[i])
            ratings[j], ratings[i] = elo_update(ratings[j], ratings[i], p_j_before_i)

    return ratings

def rank_by_elo(frame: pd.DataFrame, num_matches: int = NUM_MATCHES) -> tuple[list[int], np.ndarray]:
    texts = frame["text"].tolist()
    ratings = elo_tournament(texts, num_matches=num_matches)
    order = list(np.argsort(-ratings, kind="mergesort"))
    return order, ratings

# ------------------------------
# Validation helpers
# ------------------------------

def kendall_on_self_shuffle(frame: pd.DataFrame, num_matches: int = 3000) -> float:
    shuffled = frame.sample(frac=1.0, random_state=SEED).reset_index(drop=True)
    order, _ = rank_by_elo(shuffled, num_matches=num_matches)
    predicted_positions = np.empty(len(shuffled), dtype=int)
    for rank, idx in enumerate(order, start=1):
        predicted_positions[idx] = rank
    true_positions = shuffled["page"].to_numpy(dtype=int)
    tau = kendalltau(true_positions, predicted_positions).correlation
    return float(tau)

def validate_order(frame: pd.DataFrame, order: list[int]) -> None:
    if len(order) != len(frame):
        raise ValueError("Order length does not match frame length.")
    if len(set(order)) != len(order):
        raise ValueError("Order contains duplicate indices.")
    pages = frame.iloc[order]["page"].to_numpy(dtype=int)
    if len(set(pages.tolist())) != len(pages):
        raise ValueError("Output pages are not unique.")

def save_submission(frame: pd.DataFrame, order: list[int], out_path: Path) -> None:
    validate_order(frame, order)
    submission = page_order(frame, order)
    submission.to_csv(out_path, index=False)

# ------------------------------
# Run tournament on test books
# ------------------------------

print("Running Elo tournament on Book A...")
book_a_order, book_a_ratings = rank_by_elo(book_a_df, num_matches=NUM_MATCHES)
save_submission(book_a_df, book_a_order, BOOK_A_OUT)

print("Running Elo tournament on Book B...")
book_b_order, book_b_ratings = rank_by_elo(book_b_df, num_matches=NUM_MATCHES)
save_submission(book_b_df, book_b_order, BOOK_B_OUT)

# ------------------------------
# Diagnostics
# ------------------------------

print("Running self-shuffle validation on training set...")
train_tau = kendall_on_self_shuffle(train_df, num_matches=3000)
print(f"Train self-shuffle Kendall tau: {train_tau:.4f}")

print(f"Book A ratings range: {book_a_ratings.min():.2f} to {book_a_ratings.max():.2f}")
print(f"Book B ratings range: {book_b_ratings.min():.2f} to {book_b_ratings.max():.2f}")
print(f"Saved: {BOOK_A_OUT}")
print(f"Saved: {BOOK_B_OUT}")
print("Done.")

Loading data...
Loading cross-encoder...


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: cross-encoder/stsb-roberta-base
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Fine-tuning cross-encoder on training pairs...


Step,Training Loss


Running Elo tournament on Book A...
Running Elo tournament on Book B...
Running self-shuffle validation on training set...
Train self-shuffle Kendall tau: 0.0634
Book A ratings range: 1180.70 to 1216.22
Book B ratings range: 1181.24 to 1222.23
Saved: /teamspace/studios/this_studio/QA/submissions/BookA_rating.csv
Saved: /teamspace/studios/this_studio/QA/submissions/BookB_rating.csv
Done.
